# Query Rewriting RAG Pipeline
### PDF → Chunking → FAISS → Rewrite Query → Retrieve → Answer

In [10]:
!pip install langchain langchain-community langchain-ollama langchain-text-splitters faiss-cpu pypdf -q


[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [11]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_ollama import OllamaEmbeddings, ChatOllama

## Step 1: Load PDF

In [12]:
loader = PyPDFLoader("A._P._J._Abdul_Kalam.pdf")
pages = loader.load()

print(f"Total Pages: {len(pages)}")
print(f"Sample Content:\n{pages[0].page_content[:500]}")

Total Pages: 30
Sample Content:
A. P. J. Abdul Kalam
Official portrait, c. 2002
President of India
In office
25 July 2002 – 25 July 2007
Prime Minister Atal Bihari Vajpayee
Manmohan Singh
Vice President Krishan Kant
Bhairon Singh Shekhawat
Preceded by K. R. Narayanan
Succeeded by Pratibha Patil
Principal Scientific Adviser to the
Government of India
In office
November 1999 – November 2001
President K. R. Narayanan
Prime Minister Atal Bihari Vajpayee
Preceded by Office established
Succeeded by Rajagopala Chidambaram
Director Ge


## Step 2: Split into Chunks

In [13]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = text_splitter.split_documents(pages)

print(f"Total Chunks: {len(chunks)}")
print(f"Sample Chunk:\n{chunks[0].page_content}")

Total Chunks: 136
Sample Chunk:
A. P. J. Abdul Kalam
Official portrait, c. 2002
President of India
In office
25 July 2002 – 25 July 2007
Prime Minister Atal Bihari Vajpayee
Manmohan Singh
Vice President Krishan Kant
Bhairon Singh Shekhawat
Preceded by K. R. Narayanan
Succeeded by Pratibha Patil
Principal Scientific Adviser to the
Government of India
In office
November 1999 – November 2001
President K. R. Narayanan
Prime Minister Atal Bihari Vajpayee
Preceded by Office established
Succeeded by Rajagopala Chidambaram
Director General of Defence Research and
Development Organisation
In office
1992–1999
A. P. J. Abdul Kalam
Avul Pakir Jainulabdeen Abdul Kalam (/ˈʌbdʊl
kəˈlɑːm/ ⓘ UB-duul k ə -LAHM; 15 October 1931 –
27 July 2015) was an Indian aerospace scientist and
statesman who served as the president of India from
2002 to 2007.
Born and raised in a Muslim family in Rameswaram,
Tamil Nadu, Kalam studied physics and aerospace
engineering. He spent the next four decades as a


## Step 3: Create Embeddings & Store in FAISS

In [14]:
embeddings = OllamaEmbeddings(model="nomic-embed-text:latest")

vector_store = FAISS.from_documents(chunks, embeddings)

print(f"Vector Store Created with {vector_store.index.ntotal} vectors")

Vector Store Created with 136 vectors


## Step 4: Create Retriever

In [15]:
retriever = vector_store.as_retriever(search_kwargs={"k": 3})

## Step 5: Rewrite Query using LLM

In [16]:
llm = ChatOllama(model="llama3.2:3b", temperature=0)

# Simple list memory to store chat history
chat_history = []

def rewrite_query(original_query):
    # Include chat history as context for rewriting
    history_context = ""
    if chat_history:
        history_context = "Previous conversation:\n"
        for role, msg in chat_history:
            history_context += f"{role}: {msg}\n"
        history_context += "\n"

    rewrite_prompt = f"""{history_context}Rewrite the following query to make it more specific and precise for searching a knowledge base.
Use the conversation history (if any) to resolve references like "he", "she", "it", "they", etc.
Return ONLY the rewritten query, nothing else.

Original query: {original_query}
Rewritten query:"""
    
    response = llm.invoke(rewrite_prompt)
    return response.content.strip()

In [17]:
# Test with a vague query
original_query = "Tell me about the missile man"
rewritten_query = rewrite_query(original_query)

print(f"Original Query:  {original_query}")
print(f"Rewritten Query: {rewritten_query}")

Original Query:  Tell me about the missile man
Rewritten Query: Tell me about the individual who is referred to as the "Missile Man" in the context of the US military's space program.


## Step 6: Retrieve & Answer using Rewritten Query

In [18]:
# Retrieve using rewritten query
retrieved_docs = retriever.invoke(rewritten_query)

# Build context from retrieved chunks
context = "\n\n".join([doc.page_content for doc in retrieved_docs])

# Answer using original query (user's intent) + retrieved context
prompt = f"""Answer the question based only on the following context:

{context}

Question: {original_query}
Answer:"""

response = llm.invoke(prompt)
answer = response.content

# Store in chat history
chat_history.append(("User", original_query))
chat_history.append(("Assistant", answer))

print("Answer:", answer)

Answer: The "Missile Man of India" is a nickname given to Dr. A.P.J. Abdul Kalam, a renowned Indian scientist and engineer. He was instrumental in the development of India's ballistic missile and launch vehicle technology, and played a key role in the country's civilian space program and military missile development efforts.


## Retrieved Source Chunks

In [19]:
for i, doc in enumerate(retrieved_docs):
    print(f"\n--- Chunk {i+1} (Page {doc.metadata['page'] + 1}) ---")
    print(doc.page_content[:300])


--- Chunk 1 (Page 1) ---
2002 to 2007.
Born and raised in a Muslim family in Rameswaram,
Tamil Nadu, Kalam studied physics and aerospace
engineering. He spent the next four decades as a
scientist and science administrator, mainly at the
Defence Research and Development Organisation
(DRDO) and Indian Space Research Organisat

--- Chunk 2 (Page 4) ---
US$780 million in 2023) for the project titled Integrated Guided Missile Development Programme
(IGMDP) and appointed Kalam as its chief executive.[28] Kalam played a major role in the development
of missiles including Agni, an intermediate range ballistic missile and Prithvi, the tactical surface-to

--- Chunk 3 (Page 3) ---
career, he was involved in the design of small
hovercraft, and remained unconvinced by his choice
of a job at DRDO.[20] Later, he joined the Indian
National Committee for Space Research, working under renowned space scientist Vikram Sarabhai.[3] He
was interviewed and recruited into Indian Space Res


Test Questions

1. "Who's the missle man of india?"
2. "Tell me about when he died"
3. "What stuff did he write?"
4. "How did he become president?"
5. "What was the deal with that nuclear test?"

## Test Follow-up Query (uses chat history for context)

In [20]:
# Follow-up query - "he" should resolve to Abdul Kalam using chat history
followup_query = "When did he die?"
rewritten_followup = rewrite_query(followup_query)

print(f"Original Query:  {followup_query}")
print(f"Rewritten Query: {rewritten_followup}")
print(f"\nChat History: {chat_history}")

# Retrieve and answer
retrieved_docs = retriever.invoke(rewritten_followup)
context = "\n\n".join([doc.page_content for doc in retrieved_docs])

prompt = f"""Answer the question based only on the following context:

{context}

Question: {followup_query}
Answer:"""

response = llm.invoke(prompt)
answer = response.content

chat_history.append(("User", followup_query))
chat_history.append(("Assistant", answer))

print(f"\nAnswer: {answer}")

Original Query:  When did he die?
Rewritten Query: When did Dr. A.P.J. Abdul Kalam pass away?

Chat History: [('User', 'Tell me about the missile man'), ('Assistant', 'The "Missile Man of India" is a nickname given to Dr. A.P.J. Abdul Kalam, a renowned Indian scientist and engineer. He was instrumental in the development of India\'s ballistic missile and launch vehicle technology, and played a key role in the country\'s civilian space program and military missile development efforts.')]

Answer: 27 July 2015.
